# Excel data processing 
This file goes through the process of reading the excel file, extracting and transforming the data and saving the data into the target S3 folder in the parquet format partitioned by year and month. The pandas library has been used for faster delivery as it is a small file. In production this can be replicated using the spark dataframe for faster processing of larger files

Importing the libraries

In [2]:
%additional_python_modules openpyxl 
import pandas as pd

You are already connected to a glueetl session 582db990-4286-4371-b19b-ff20ef10f0a9.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Additional python modules to be included:
openpyxl



Reading the data using pandas

In [6]:
df = pd.read_excel("s3://cn01-project-input-205096516800-us-east-2-an/distance_log_excel_files/Distancelog1.xlsx")
df.head(5)

     Origin Destination  ...  Distance_km                 Date
0   Brandon   Saskatoon  ...          659                  NaN
1  Winnipeg     Brandon  ...          215  2025-11-30 00:00:00
2  Edmonton     Kelowna  ...          575           03-06-2016
3   Lankey      Burnaby  ...           30  2020-08-23 00:00:00
4   Brandon      Barrie  ...         1700                  NaN

[5 rows x 6 columns]


Renaming columns and splitting to get the city and province level details separately

In [7]:
column_names = ["trip_origin","trip_destination","start_odometer","end_odometer","distance","trip_date"]
df.columns=column_names
df[['trip_origin_city', 'trip_origin_province']] = df["trip_origin"].str.split(', ', expand=True)
df[['trip_destination_city', 'trip_destination_province']] = df["trip_destination"].str.split(', ', expand=True)

Defining a schema of columns that is required for the target data warehouse and adding columns if not already present

In [8]:
df=df.drop(columns = ['trip_origin','trip_destination'])
schema = ["trip_date","start_odometer","end_odometer","distance","AB_kms", "BC_kms", "SK_kms", "MB_kms", "ON_kms",
       "QC_kms", "YT_kms","total_fuel","trip_origin_city","trip_origin_province","trip_destination_city","trip_destination_province","vin_number"]

In [11]:
for col in schema:
        if col not in df.columns:
            df[col] = pd.NA

df =df[schema]
df[["AB_kms", "BC_kms", "SK_kms", "MB_kms", "ON_kms",
       "QC_kms", "YT_kms","total_fuel"]] = df[["AB_kms", "BC_kms", "SK_kms", "MB_kms", "ON_kms",
       "QC_kms", "YT_kms","total_fuel"]].fillna(0)

Creating a dictionary of city values to map to correct names and formatting the columns for correct data types

In [12]:
city_map = {"Edmanton" : "Edmonton", "Lankey ":"Langley", "Lankey":"Langley"}
df['trip_origin_city'] = df['trip_origin_city'].replace(city_map)
df['trip_destination_city'] = df['trip_destination_city'].replace(city_map)
df['trip_date'] = pd.to_datetime(df['trip_date'])
df['start_odometer'] = df['start_odometer'].astype(float)
df['end_odometer'] = df['end_odometer'].astype(float)
df['distance'] = df['distance'].astype(float)
df['AB_kms'] = df['AB_kms'].astype(float)
df['BC_kms'] = df['BC_kms'].astype(float)
df['SK_kms'] = df['SK_kms'].astype(float)
df['MB_kms'] = df['MB_kms'].astype(float)
df['ON_kms'] = df['ON_kms'].astype(float)
df['QC_kms'] = df['QC_kms'].astype(float)
df['YT_kms'] = df['YT_kms'].astype(float)
df['total_fuel'] = df['total_fuel'].astype(float)


Extracting year and month columns as well as setting trip_date to a default date for testing purposes

In [13]:
df['trip_date'] = df['trip_date'].fillna('2025-11-30')
df['year'] = df['trip_date'].dt.year
df['month'] = df['trip_date'].dt.month
df['trip_date'] = df['trip_date'].dt.date

Creating a city to province mapping to ensure province level details are entered for each record.

In [ ]:
city_state_map = {"Edmonton":"AB","Caledon":"ON","Willowdale":"ON","Kelowna":"BC","Langley":"BC","Sudbury":"ON","Brooks":"BC","Burnaby":"BC","Brandon":"ON","Red Deer":"AB","Saskatoon":"SK","Consart":"AB",
"Prince Albert":"SK","Barrie":"ON","Regina":"SK","Winnipeg":"MB","Kingston":"ON","Thunder Bay":"ON","Kamloops":"BC"}
df['trip_origin_province'] = df['trip_origin_city'].map(city_state_map)
df['trip_destination_province'] = df['trip_destination_city'].map(city_state_map)


df['trip_origin_city'] = df['trip_origin_city'].str.lower().str.strip()
df['trip_destination_city'] = df['trip_destination_city'].str.lower().str.strip()

df.duplicated().sum()

# 21 rows missing vin_number
df.isnull().sum()

Writing data to target as partitioned file

In [ ]:
# df.to_parquet(path="s3://cn01-project-output-205096516800-us-east-2-an/output_parquet_files/distance_logs/",partition_cols=['year','month'])

In [15]:
pd.set_option('display.max_columns', None)
print(df.head(5))

    trip_date  start_odometer  end_odometer  distance  AB_kms  BC_kms  SK_kms  \
0  2025-11-30        100000.0      100659.0     659.0     0.0     0.0     0.0   
1  2025-11-30         62845.0       63060.0     215.0     0.0     0.0     0.0   
2  2016-03-06         92604.0       93179.0     575.0     0.0     0.0     0.0   
3  2020-08-23         41908.0       41938.0      30.0     0.0     0.0     0.0   
4  2025-11-30        128412.0      130112.0    1700.0     0.0     0.0     0.0   

   MB_kms  ON_kms  QC_kms  YT_kms  total_fuel trip_origin_city  \
0     0.0     0.0     0.0     0.0         0.0          brandon   
1     0.0     0.0     0.0     0.0         0.0         winnipeg   
2     0.0     0.0     0.0     0.0         0.0         edmonton   
3     0.0     0.0     0.0     0.0         0.0          langley   
4     0.0     0.0     0.0     0.0         0.0          brandon   

  trip_origin_province trip_destination_city trip_destination_province  \
0                   ON             saskato

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   trip_date                  21 non-null     object 
 1   start_odometer             21 non-null     float64
 2   end_odometer               21 non-null     float64
 3   distance                   21 non-null     float64
 4   AB_kms                     21 non-null     float64
 5   BC_kms                     21 non-null     float64
 6   SK_kms                     21 non-null     float64
 7   MB_kms                     21 non-null     float64
 8   ON_kms                     21 non-null     float64
 9   QC_kms                     21 non-null     float64
 10  YT_kms                     21 non-null     float64
 11  total_fuel                 21 non-null     float64
 12  trip_origin_city           21 non-null     object 
 13  trip_origin_province       21 non-null     object 
 